# Approve paid MCP tool calls in the Agents SDK

Some tools have consequences a person should sign off on: they charge a card, send email, or contact someone outside your company. The Agents SDK can pause a run right before such a tool call, hand you the exact arguments, and continue only after you approve.

This notebook shows that pattern on a remote MCP server whose tools spend real money in production. [Instant Expert](https://instant.expert) lets an agent pay specific people (executives, operators, domain experts) for a short call or a written answer, charged only if they accept. We use its test mode, so the notebook runs with no account: searches return fictional people, no email is sent, and the order goes on a Stripe test card.

You'll learn how to:

- connect a Streamable HTTP MCP server with `MCPServerStreamableHttp` and a bearer token,
- require approval for one tool with `require_approval`, then review and approve the run's interruptions,
- give the agent the instructions the MCP server sends during its handshake,
- wait for long-running background jobs with a function tool, instead of spending model turns on polling.

## Setup

You need Python 3.10 or newer and an `OPENAI_API_KEY`. The model calls cost a few cents. Instant Expert's test mode is free and needs no sign-up.

In [ ]:
%pip install --quiet -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


Running the rest of the notebook calls two live services, so it's opt-in: set `RUN_LIVE=1` in your environment, or change the `RUN_LIVE` line below to `True`.

Test mode takes an anonymous sandbox token from one `POST` request. It lasts 24 hours, works only on the test server, and each network can create five an hour. We print the server URL and expiry, not the token.

In [ ]:
import json
import os
import urllib.request

# Opt in to calling the OpenAI API (a few cents of model usage) and Instant
# Expert's free test server.
RUN_LIVE = os.environ.get("RUN_LIVE") == "1"
if not RUN_LIVE:
    raise RuntimeError("Set RUN_LIVE=1 (or RUN_LIVE = True here) to run the rest of this notebook.")
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY first"

request = urllib.request.Request("https://instant.expert/api/sandbox", method="POST", data=b"")
with urllib.request.urlopen(request) as response:
    sandbox = json.load(response)

print(sandbox["mcp_url"], "until", sandbox["expires_at"])

https://instant.expert/mcp/test until 2026-10-10T06:26:52.441Z


## Connect the server and require approval for the paid tool

`require_approval` sets a policy per tool name. Here `send_requests`, the tool that places the order and charges the card, always needs approval; every other tool (searching, drafting, previewing the order) runs freely.

The timeouts are raised from the SDK's 5-second default because some of this server's tools take 10 to 20 seconds.

In [ ]:
from agents.mcp import MCPServerStreamableHttp

server = MCPServerStreamableHttp(
    name="instant_expert",
    params={
        "url": sandbox["mcp_url"],
        "headers": {"Authorization": f"Bearer {sandbox['token']}"},
        "timeout": 120,
        "sse_read_timeout": 120,
    },
    client_session_timeout_seconds=120,
    cache_tools_list=True,
    require_approval={"send_requests": "always"},
)
await server.connect()

print([tool.name for tool in await server.list_tools()])

['search_people', 'import_people', 'queue_requests', 'plan_outreach', 'get_job', 'get_search', 'list_searches', 'get_request_draft', 'prepare_request_order', 'send_requests', 'get_request_order', 'list_requests', 'get_request', 'simulate_response']


## Use the server's own instructions

An MCP server can send instructions during the `initialize` handshake. The SDK keeps them in `server.server_initialize_result` but doesn't add them to the agent's prompt, so pass them in yourself. This server's instructions describe its workflow (search, wait for the job, draft, preview the order, then send) and start with a TEST MODE paragraph.

In [ ]:
server_instructions = server.server_initialize_result.instructions
print(server_instructions[:500], "...")

TEST MODE. This connection uses the buyer's sandbox: searches return fictional people (emails on the reserved .example domain), no email is ever sent (each would-be email is recorded and shown in get_request), orders use a Stripe test card that is attached automatically, and every response carries test_mode: true. Each fictional person answers on their own right after delivery (books, replies, declines, never answers, or bounces); call simulate_response to force any outcome now, including call_c ...


## Wait for background jobs in code

Searching and drafting run as background jobs: the tool returns a `job_id` and the client polls `get_job`. If the model does the polling, every check costs a turn. One function tool that polls in Python keeps the run short. It calls the MCP server directly with `server.call_tool`.

In [ ]:
import asyncio

from agents import function_tool

FINISHED = {"succeeded", "failed", "cancelled"}


@function_tool
async def wait_for_job(job_id: str) -> str:
    """Wait for a search, import or draft job to finish and return the final job.

    Args:
        job_id: The job_id returned by search_people, import_people or queue_requests.
    """
    for _ in range(150):  # about five minutes at the server's suggested interval
        result = await server.call_tool("get_job", {"job_id": job_id})
        job = json.loads(result.content[0].text)
        if job["status"] in FINISHED:
            break
        await asyncio.sleep(job.get("poll_after_seconds") or 2)
    return json.dumps(job)

## Run the agent

The prompt tells the model that sending pauses for approval, so it calls `send_requests` as soon as the order preview is ready instead of stopping to ask in chat. Parallel tool calls are off so the model reads the search results before it drafts; with them on, it can draft in the same turn as the search.

In [ ]:
from agents import Agent, ModelSettings, Runner

MODEL = "gpt-5.4-mini"

ROLE = (
    "You help a founder reach the right executives. Work one step at a time: search, "
    "wait for the job with wait_for_job, read the people with get_search, then draft "
    "from that search_id. If a search finds fewer people than asked, continue with the "
    "ones it found. Once the order preview is ready, call send_requests right away. "
    "Don't ask for approval in chat: calling send_requests is how you ask, because the "
    "run pauses there and shows the founder the order. After sending, say who was "
    "invited and at what price."
)

agent = Agent(
    name="Expert finder",
    model=MODEL,
    instructions=f"{ROLE}\n\n{server_instructions}",
    mcp_servers=[server],
    tools=[wait_for_job],
    model_settings=ModelSettings(parallel_tool_calls=False),
)

task = (
    "Find 3 sales leaders at software companies (if the search finds fewer, use the "
    "ones it found). Offer each of them $40 for a 15-minute call about how their team "
    "evaluates new sales tools, with a $150 total budget. Prepare the order, then send it."
)
result = await Runner.run(agent, task, max_turns=40)

print(f"{len(result.interruptions)} tool call(s) waiting for approval")

1 tool call(s) waiting for approval


## Review the order

Each interruption is a `ToolApprovalItem` with the tool's name and the exact arguments the model wants to send. Show a person what those arguments will do. Here we look up the draft behind the call and print who gets invited, what each person receives, what the buyer pays if they accept, and the total cap.

In [ ]:
def dollars(cents: int) -> str:
    """Format cents exactly, so the approval screen never rounds the amount."""
    return f"${cents / 100:,.2f}"


draft_ids, invited = [], 0
for item in result.interruptions:
    arguments = json.loads(item.arguments)
    draft_ids.append(arguments["draft_id"])
    draft_result = await server.call_tool("get_request_draft", {"draft_id": arguments["draft_id"]})
    draft = json.loads(draft_result.content[0].text)
    print(f"{item.name}: {draft['message']}")
    for person in draft["recipients"]:
        print(
            f"  {person['name']}: receives {dollars(person['offer_cents'])}, "
            f"you pay {dollars(person['price_with_fee_cents'])} if they accept"
        )
    print(f"  Budget cap: {dollars(draft['max_spend_cents'])}")
    invited += len(draft["recipients"])

send_requests: I’d like to invite you to a 15-minute call to learn how your team evaluates new sales tools.
  Naveen Yilmaz: receives $40.00, you pay $50.00 if they accept
  Caleb Dalca: receives $40.00, you pay $50.00 if they accept
  Budget cap: $150.00


## Approve or reject, then resume

`result.to_state()` captures the paused run. Approving lets the tool call run; rejecting sends the model a refusal instead, and it continues without placing the order. Then run again from the state. In an app, the decision would come from a button or a chat reply; here it's a variable.

In [ ]:
approve_order = True  # Review the order above first.

state = result.to_state()
for item in result.interruptions:
    if approve_order:
        state.approve(item)
    else:
        state.reject(item)

result = await Runner.run(agent, state, max_turns=40)
print(result.final_output)

Sent the order in test mode.

Invited 2 people:
- Naveen Yilmaz — $40 offer, $50 cost to you if he accepts
- Caleb Dalca — $40 offer, $50 cost to you if he accepts

Details:
- Call length: 15 minutes
- Total budget cap: $150
- Payment mode: test
- Follow-ups: 2 and 4 days
- Delivery is still pending in the background

Note: the search only found 2 matches, so I used the ones it found.


## Check the outcomes

`send_requests` returns once the order is saved; delivery finishes in the background. In test mode, each fictional person then responds on their own: some book a call, some reply with a question, some decline, and some never answer. A booked call charges the test card for that person only. This cell reads the results directly from the server, without the model.

In [ ]:
if approve_order and draft_ids:
    for _ in range(30):
        order_result = await server.call_tool("get_request_order", {"draft_id": draft_ids[0]})
        if json.loads(order_result.content[0].text)["delivery"]["complete"]:
            break
        await asyncio.sleep(2)

    # Newest first, so the first rows are this order's requests.
    listing = await server.call_tool("list_requests", {"limit": invited})
    for request in json.loads(listing.content[0].text)["items"]:
        if request["kind"] == "request":
            print(f"{request['recipients'][0]['name']}: {request['status']}")

Naveen Yilmaz: booked
Caleb Dalca: sent


When you're done, close the connection.

In [ ]:
await server.cleanup()

## Going further

- **Approve by rule.** Besides a map of tool names, `require_approval` accepts `{"always": {"tool_names": [...]}}` lists or a function that decides per call. `state.approve(item, always_approve=True)` also approves later calls to the same tool in the run, and `state.reject(item, rejection_message="...")` tells the model why it was refused. See the Agents SDK guide to [human in the loop](https://openai.github.io/openai-agents-python/human_in_the_loop/).
- **Real accounts.** Point the server at `https://instant.expert/mcp`. It accepts OAuth access tokens only, so instead of `headers`, pass an OAuth provider such as FastMCP's `fastmcp.client.auth.OAuth` as `params["auth"]`; it signs in through the browser once. Sending from an agent also needs paid sending turned on for that connection and a saved card. Details are in the [Instant Expert MCP docs](https://instant.expert/docs/mcp).
- **Other frameworks.** The same server works from LangChain, CrewAI and the JavaScript Agents SDK; examples are in [Instant-Expert/integrations](https://github.com/Instant-Expert/integrations).